In [5]:
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
client = OpenAI()

In [6]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [f.parse() for f in reader.read()]
len(documents)

72

In [7]:
from minsearch import Index

index = Index(text_fields=["content"], keyword_fields=["filename"])
index.fit(documents)

query = "How does the agentic loop keep calling the model until it stops?"
results = index.search(query, num_results=5)
results[0]["filename"]

'01-agentic-rag/lessons/14-agentic-loop.md'

In [8]:
instructions = "Answer the question using the provided context. If it's not in the context, say you don't know."

def build_context(results):
    return "\n\n".join(f"{d['filename']}\n{d['content']}" for d in results)

def rag(idx, query, model="gpt-5.4-mini"):
    results = idx.search(query, num_results=5)
    prompt = f"QUESTION: {query}\n\nCONTEXT:\n{build_context(results)}"
    return client.responses.create(
        model=model,
        input=[
            {"role": "developer", "content": instructions},
            {"role": "user", "content": prompt},
        ],
    )

resp = rag(index, query)
resp.usage.input_tokens

7083

In [9]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)
len(chunks)

295

In [10]:
chunk_index = Index(text_fields=["content"], keyword_fields=["filename"])
chunk_index.fit(chunks)

resp_chunk = rag(chunk_index, query)
print(resp_chunk.usage.input_tokens)
print(resp.usage.input_tokens / resp_chunk.usage.input_tokens)

2266
3.1257722859664607


In [11]:
calls = []

def search(query: str) -> list:
    """Search the course lessons for passages matching the query."""
    calls.append(query)
    return chunk_index.search(query, num_results=5)

from toyaikit.tools import Tools
from toyaikit.llm import OpenAIClient
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

agent_tools = Tools()
agent_tools.add_tool(search)

agent_instructions = "You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering."

chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=agent_instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini"),
)

runner.loop(prompt="How does the agentic loop work, and how is it different from plain RAG?", callback=callback)
len(calls)

-> Response received


-> Response received


4